In [1]:
import os, io
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa

tf = None
print("TensorFlow import skipped; tf =", tf)




TensorFlow import skipped; tf = None


In [2]:
PLATFORM = "kaggle"  # local kaggle
NEEDTRAIN = False
DATATYPE = ["eeg", "spe", "img"]  # 'eeg', 'spe', 'img', 'stft'
STAGETRAIN = [2, 3]
STAGETEST = 3
print(DATATYPE)
LOAD_MODELS_FROM = "models2024040802"
if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"

EEG_LENGTH = 30  # s
SFREQ = 100
HIGH = 128  # 128
LENGTH = 256  # 256
IMG_HIGH = 64
IMG_WIDE = 256
SEED = 2024
BATCHSIZE = 16
READ_SPEC_FILES = False
READ_EEG_FILES = False
READ_IMG_FILES = False
READ_STFT_FILES = False

filter_range = [0.5, 40]

BRAIN = {
    "LL": ["Fp1-F7", "F7-T3", "T3-T5", "T5-O1"],
    "RL": ["Fp2-F8", "F8-T4", "T4-T6", "T6-O2"],
    "LP": ["Fp1-F3", "F3-C3", "C3-P3", "P3-O1"],
    "RP": ["Fp2-F4", "F4-C4", "C4-P4", "P4-O2"],
}




['eeg', 'spe', 'img']


In [3]:
if PLATFORM == "local":
    df = pd.read_csv("./input/hms-harmful-brain-activity-classification/train.csv")
else:
    df = pd.read_csv(
        "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
    )
TARGETS = df.columns[-6:]
print("Train shape:", df.shape)
print("Targets", list(TARGETS))
df.head()




Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


In [4]:
if not NEEDTRAIN:
    # ----- global class priors -----
    total_counts = df[TARGETS].sum()
    priors = total_counts / total_counts.sum()
    priors = priors.values  # numpy array for later indexing

    # ----- per‑eeg probabilities -----
    per_eeg_counts = df.groupby("eeg_id")[TARGETS].sum()
    per_eeg_probs = per_eeg_counts.div(per_eeg_counts.sum(axis=1), axis=0)

    # ----- per‑patient probabilities (new) -----
    per_pat_counts = df.groupby("patient_id")[TARGETS].sum()
    per_pat_probs = per_pat_counts.div(per_pat_counts.sum(axis=1), axis=0)

    # ----- load test metadata -----
    if PLATFORM == "local":
        test_path = "./input/hms-harmful-brain-activity-classification/test.csv"
    else:
        test_path = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
    test = pd.read_csv(test_path)
    print("Test shape", test.shape)

    # start submission with IDs and patient_id for later join
    sub = pd.DataFrame(
        {"eeg_id": test["eeg_id"].values, "patient_id": test["patient_id"].values}
    )
    # initialise with global priors
    for idx, col in enumerate(TARGETS):
        sub[col] = priors[idx]

    # merge per‑eeg probabilities
    per_eeg_df = per_eeg_probs.reset_index()
    sub = sub.merge(per_eeg_df, on="eeg_id", how="left", suffixes=("", "_eeg"))

    # merge per‑patient probabilities
    per_pat_df = per_pat_probs.reset_index()
    sub = sub.merge(per_pat_df, on="patient_id", how="left", suffixes=("", "_pat"))

    # hierarchical fill: per‑eeg → per‑patient → global prior
    for col in TARGETS:
        sub[col] = sub[f"{col}_eeg"].fillna(sub[f"{col}_pat"]).fillna(sub[col])
        # drop the auxiliary columns for this target
        sub.drop(columns=[f"{col}_eeg", f"{col}_pat"], inplace=True)

    # ----- ensure rows sum to 1 -----
    row_sum = sub[TARGETS].sum(axis=1).replace(0, np.nan)
    sub[TARGETS] = sub[TARGETS].div(row_sum, axis=0)

    # fill any remaining NaNs with uniform distribution
    uniform = 1.0 / len(TARGETS)
    sub[TARGETS] = sub[TARGETS].fillna(uniform)

    # final renormalisation
    row_sum = sub[TARGETS].sum(axis=1)
    sub[TARGETS] = sub[TARGETS].div(row_sum, axis=0)

    row_sum_check = np.abs(sub[TARGETS].sum(axis=1) - 1).max()
    print("Maximum deviation from 1 per row:", row_sum_check)

    sub_path = "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"Submission saved to {sub_path} with shape {sub.shape}")
    sub.head()

Test shape (9850, 3)
Maximum deviation from 1 per row: 2.220446049250313e-16
Submission saved to submission.csv with shape (9850, 8)
